# The physics of a synthetic jam — what TrafficFlowBench's data does and doesn't contain

This notebook is a **research log, not a pipeline**. Over two days a pharmacist with a traffic-physics intuition and an AI assistant
(Claude Code) took the released data apart hypothesis by hypothesis — *jams ripple upstream, capacity collapses, the week repeats,
cars on the road set the queue length, a shock-wave formula moves the tail, ARIMA, Kalman, a learned cell-transmission model, …* —
and **measured each one**. About half of the intuitions held, with a number. The other half failed, also with a number, and the
failures are the more useful part: they say where the score can and cannot be found.

Every figure and table below comes from a re-runnable script over the full train split (10 corridors, 273 days, 2.1 M eligible
cells per corridor-month). Nothing is tuned on the leaderboard; the public score is used only as the final arbiter, and the
leaderboard decomposition at the end explains exactly where our 0.79 → 0.84 came from and where the gap to the public 0.865 sits.

**Contents**
1. What one corridor-day looks like (and what we are asked to predict)
2. Anatomy of a jam: fixed head, creeping tail, flat plateaus, no waves
3. The fundamental diagram in the data; where a road actually breaks down
4. Cars on the road ↔ jam length: a two-parameter law that validates on March and April
5. What forecasts a jam and what doesn't (load, density, derivatives, pointer ratios, the window selection)
6. Month to month: the typical week, March/April anomalies, bottlenecks that switch on and off
7. Learning the dynamics: regressions, Kalman, learned CTM, ARIMA, space-time patches
8. Scoreboard of hypotheses, and the leaderboard decomposition

In [ ]:
from pathlib import Path
from IPython.display import Image, display
def figpath(name):
    hits = [p for p in Path('/kaggle/input').glob(f'**/{name}') if 'jam-physics' in str(p)] or list(Path('/kaggle/input').glob(f'**/{name}'))
    assert hits, f'figure {name} not found under /kaggle/input — attach the dataset plarmuseau/trafficflowbench-jam-physics-figs'
    return str(hits[0])
print('figures at', Path(figpath('fd_D12_I5_N.png')).parent)

## 1. What one corridor-day looks like

Ten directional freeway corridors, one detector per link (0.1–0.4 km apart), a record every 5 minutes: speed, flow, occupancy,
and a quality field. Task 1 blanks 20/30/50 % of the eligible cells of each day at random and asks for speed and flow there;
Task 2 gives 60 minutes of history in 160 windows and asks *queued yes/no* per link for the next 30 minutes; Task 3 checks vehicle
conservation on the Task-1 answer; Task 4 is an OD inverse problem. **Nothing asks for a day or a month ahead** — the only forecast
is 30 minutes, inside each window.

Below: I-5 northbound (D12_I5_N, 71 km, 258 links) on 23 January 2031. Left the complete train layer, middle what a participant
actually receives on an R3 day (half the cells grey), right the flow per lane. Traffic drives left → right, time runs down.

In [ ]:
display(Image(filename=figpath('D12_I5_N_2031-01-23_timespace.png'), width=1100))

Two wedges: a jam born at a bottleneck near km 33 at 07:30 whose **tail grows upstream in a straight line for ten hours**, and a
second one near km 67 from noon. The block at km 0–5 recurs every day — a structural restriction at the corridor entrance, not an
incident. Flow barely shows the jams (flow inside a queue stays at ~0.75 of capacity); speed carries them.

The same day as an animation (a bar per link, colour = speed, the time-space picture building up underneath):

In [ ]:
display(Image(filename=figpath('D12_I5_N_2031-01-23.gif')))

A live version of the picture from the competition data itself (one parquet file, a few seconds):

In [ ]:
import glob, numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
root = [p for depth in range(1, 6) for p in Path('/kaggle/input').glob('/'.join(['*'] * depth) + '/corridors') if (p / 'D12_I5_N').exists()][0]
print('competition data at', root)
panel, date = 'D12_I5_N', '2031_01_23'
f = glob.glob(f'{root}/{panel}/train/mainline_states/*/synthetic_mainline_{date}.parquet')[0]
d = pd.read_parquet(f); d = d[d.is_score_eligible.astype(bool)]
topo = pd.read_csv(f'{root}/{panel}/network/lwr_mainline_topology.csv'); ml = set(topo.link_id)
mp = d.groupby('link_id').milepost.mean()
s = [np.sign(mp[o] - mp[r.link_id]) for r in topo.itertuples() for o in str(r.outgoing_link_ids).split(';') if o in ml and r.link_id in mp.index and o in mp.index]
sgn = 1.0 if np.sum(s) >= 0 else -1.0                     # travel direction from the topology; milepost is in km
order = (mp[mp.index.isin(ml)] * sgn).sort_values().index.tolist(); li = {l: i for i, l in enumerate(order)}
x = mp.reindex(order).to_numpy() * sgn; x -= x.min(); edges = np.r_[x[0] - .1, (x[1:] + x[:-1]) / 2, x[-1] + .1]
t = pd.to_datetime(d.timestamp); V = np.full((288, len(order)), np.nan)
V[(t.dt.hour * 12 + t.dt.minute // 5).to_numpy(), d.link_id.map(li).to_numpy()] = d.speed_kmh
cm = plt.get_cmap('RdYlGn').copy(); cm.set_bad('#d9d9d9')
plt.figure(figsize=(9, 6)); plt.pcolormesh(edges, np.r_[np.arange(288) / 12, 24], V, cmap=cm, vmin=0, vmax=120, shading='flat')
plt.gca().invert_yaxis(); plt.colorbar(label='speed km/h'); plt.xlabel('km along direction of travel →'); plt.ylabel('hour'); plt.title(f'{panel} {date}: speed, eligible cells'); plt.show()

## 2. Anatomy of a jam

4,497 jam episodes (8-connected blobs of cells with speed < 0.5 × the link's normal speed) over 6 corridors × 273 days:

| property | measured |
|---|---|
| episodes per day | 0.6 (D7_I10_W) … 4.5 (D12_I5_S) |
| the head | **stays put** (median drift 0 km): a fixed bottleneck; the most frequent head link carries 21–56 % of a corridor's episodes |
| where heads sit | 46 % just upstream of a lane drop (base rate 14 %), 65 % of a capacity drop (base 29 %) |
| growth | 1,519 ripple upstream (≥ 10 min growth), 2,004 appear at once, 974 stay ≤ 2 links |
| tail speed while growing | p50 ≈ 1 km/h, p90 ≈ 4 km/h (slow: inflow is only just above the bottleneck's discharge) |
| growth vs decay | not symmetric: 80 min growth vs 185 min decay (median); retreat ≈ 0.6 × growth speed; **60 % vanish in one 5-min step** while still ≥ half their maximum length |
| speed inside | ⅓ of normal, and almost constant: within-link sd 0.8 km/h over hours |
| waves inside the jam | **none**: 75–91 % of the speed variance inside jams is between links (fixed plateaus per link), the rest changes everywhere at once (same-moment correlation 0.81–0.86 between neighbours, 0.02–0.07 after 5 min in either direction) |

The plateaus are the road's own layout, not the ramps: across 511 adjacent link pairs jammed together, the capacity ratio and lane
change to the next link explain 82 % of the speed step (t = −35 and −10); on-/off-ramps explain nothing (median ramp flow 22–34 veh/h).
More lanes ahead → *slower* ahead (−11.7 km/h): the same queue flow spread over more lanes sits deeper on the congested branch.

**The derivative pictures make the fronts visible.** dv/dt is zero everywhere except on two straight lines per jam: the front moving
upstream (blue) and the resolution moving downstream (red). d²v/dt² adds nothing beyond noise; dc/dt (load) shows demand changing
along the whole corridor at once (horizontal stripes).

In [ ]:
display(Image(filename=figpath('derivatives_D12_I5_N_2031-01-23.png'), width=1100))

Denoising the front image: thresholding v (0.5 × normal), carrying missing cells from time neighbours, and a 3-step majority vote
reduce single-step flicker on this corridor from 333 to 9 transitions out of 40,767 over 273 days.

In [ ]:
display(Image(filename=figpath('fronts_D12_I5_N_2031-01-23.png'), width=1100))

The spatial difference v(k links ahead) − v(here) shows the jam coming: a free link with a jam directly ahead is jammed within
30 min in **27 %** of cases (base rate 0.6 %), with a median arrival of 10–15 min — 40× the odds, yet still 3 times out of 4 the jam
does not reach the link in time. That is why hard "jam ahead → jam here" rules lost to persistence (0.34–0.39 vs 0.40 IoU).

In [ ]:
display(Image(filename=figpath('spatial_dv_D12_I5_N_2031-01-23.png'), width=1100))

## 3. The fundamental diagram, and where a road breaks down

The data sits on the triangular diagram of `fd_parameters.csv` almost exactly: below capacity every car drives the free speed
(117.5 km/h) whatever the flow, then the state collapses onto the congested line. The "optimum speed below the limit" that a
spacing model predicts on real roads is absent here — the simulator has no curvature before the corner.

In [ ]:
display(Image(filename=figpath('fd_D12_I5_N.png'), width=1100))

Three capacities, measured on 3,693 episodes (as a fraction of the `fd_parameters` capacity, which is the record q99.9):

| | median | p25–p75 |
|---|---|---|
| max flow at the head in the 30 min before collapse | 0.75 | 0.68–0.85 |
| discharge out of the head while jammed (the sustainable capacity) | 0.69 | 0.57–0.77 |
| drop from before to after | 6 % | up to 18 % (p75) |

So a bottleneck breaks around 0.75 of the record and then passes ~0.69 — hysteresis: the queue only dissolves when inflow drops
*below* the lower discharge, which is why jams keep growing ~1.5 h after the demand peak. **Breakdown is a probability, not a
threshold**: a free link with free traffic ahead collapses within 15 min in at most 1.5 % of 5-min slots (at 85–90 % load); with a jam
already downstream the same load gives 27 %.

Cars inside jams never approach the physical standstill density: p99.9 of per-lane density is 73–81 veh/km (≈ 13 m per car),
against a k_jam of 100 in the parameters and ~140 for bumper-to-bumper.

## 4. Cars on the road ↔ jam length

The hypothesis (Paul's "model 5"): *count the cars; what exceeds the road's capacity stands in the queue.* With N = Σ density × length
over the corridor:

**jam length (km) = max(0, N − N₀) / (k_jam − k_free)**, with **N₀ = L × c_bottleneck / v_free**

Both constants come from the road parameters — no regression. Fitted hinge vs physical: slope 0.60–0.64 km per 100 cars vs
0.57–0.59 from the densities; N₀ fitted / N₀ physical = 0.84–1.04 (mean 0.95, correlation 0.978 over 10 corridors).

In [ ]:
display(Image(filename=figpath('cars_vs_jam.png'), width=1100))

| split (observed cells, train constants) | R² median over 10 corridors | MAE km |
|---|---|---|
| train (masked + filled) | 0.972 | 0.34 |
| **March** (validation) | 0.948 | 0.40 |
| **April** (private) | 0.937 | 0.50 |

The road does not change between months; only the demand does. For the *change* of the jam over the next 30 min, the change of N
over the last 30 min (R² 0.49) beats the tail's own trend (0.41). The same balance measured at the **edges of one jam** (inflow at the
tail minus discharge at the head) correlates only 0.26 with the growth — the net effect is tens of cars inside flows of ~4,000/h
with a few percent of noise each. **Conservation works summed over a corridor and fails per link**: the same lesson killed an Euler
fill of blank cells, a one-step CTM (85 % / 161 % worse than persistence) and the Rankine–Hugoniot tail speed (MAE 3.5 km vs 1.2 km
for the plain trend). The shock formula is right; the data cannot resolve Δq.

## 5. What forecasts a jam — and what doesn't

Onset precursors, 3,981 jams born at a bottleneck head vs 9,106 control moments (AUC, 0.5 = coin toss):

| signal | 5 min before | 15 min | 30 min |
|---|---|---|---|
| density of the link just downstream (k / k_crit) | **0.818** | 0.784 | 0.746 |
| density per lane, downstream link (= spacing, monotone) | 0.811 | 0.786 | **0.764** |
| load q/cap, own link | 0.792 | 0.734 | 0.693 |
| cars over 3 km upstream, N/N_c | 0.764 | 0.742 | 0.725 |
| arriving flow 20–30 km upstream, shifted by travel time | — | 0.69 (10 min) | 0.70 (15 min) |
| speed, dv/dt, d²v/dt², neighbour contrasts of these | 0.50–0.58 | | |

Median 5 min before a jam: 14 cars/km/lane (70 m per car) vs 10 (99 m). Speed itself is still 115.7 vs 116.3 km/h — the triangle has
no slope before the corner, so **speed derivatives carry nothing**; the only usable derivative is ∂k/∂t, "the cell is filling"
(AUC 0.67), i.e. the left side of the conservation law. Dissolution is not announced by anything (AUC ≤ 0.58).

**But none of this moves the onset score.** The organizers select onset windows so that a jam appears *exactly at T+30* (39 of 40
official train windows): the *when* is given, only the *where* is asked. A place × time-of-day memory of where jams are born
(±30 min, top-4 links at T+30) scores IoU 0.60; load or density multiplied in gives 0.60–0.59. Timing signals are redundant with
the window selection.

In [ ]:
display(Image(filename=figpath('flow_source.png'), width=900))

Where flow *changes* come from (Feb, correlation of Δq with Δq at an offset 5 and 10 min earlier): in free flow from **9 km
upstream after 5 min, 19 km after 10 min** — exactly the distance cars drive (117 km/h); in a jam from **0.75 km downstream after
5 min, 1.75 km after 10 min** — a wave moving upstream at ~9–10 km/h. Paul's "in free flow the predecessor sets the flow, in a jam
the zone ahead does" is in the data; it is just not enough to forecast a 0/1 state 30 minutes out.

## 6. Month to month

A typical week (weekday × 15 min × position, 273 days) explains **90 %** of the variance of load but only **52 %** of speed — speed is
flat except inside the wedges, and the wedges' timing is the day-to-day part. With the jam state known, two levels per link (free /
jammed) explain 97 % of log speed (rest 2.4 km/h ≈ detector noise), and that holds in March (96 %) and April (96 %): the physics
transfers; the typical week doesn't (58 % → 37–43 %).

In [ ]:
display(Image(filename=figpath('typical_week_D12_I5_N.png'), width=1000))

Weekday matters only in the morning (weekday commute at 6–10 h; weekends start late) — the afternoon peak is identical on all seven
days (peak flow 6,077 / 6,121 / 6,115 veh/h on weekday / Saturday / Sunday), so weekend afternoons jam just as much. Synthetic.

April vs the typical week (weekday-matched, observed cells only): whole bottlenecks switch off for a month (D12_I5_N km 35.4:
24 % of the time jammed in train → 0 % in March and April; D7_I210_W km 38: 26 % → 0 %) and new ones appear (D12_I5_S km 1.6: 0 % →
35 %). The discharge of a head that exists in both periods is identical (0.64/0.64, 0.66/0.66, 0.58/0.58 of the record): **the road's
capacity does not change; the month's demand distribution picks which point tips first.**

In [ ]:
display(Image(filename=figpath('anomaly_private_mean.png'), width=1200))

Does March predict April? Per link, April's jam share from train: RMSE 5.2 pp; from March: 6.5 pp; the shifts of March and April
relative to train correlate −0.07. On three corridors with real structural shifts March is much better (D12_I5_N 9.5 → 5.6), on seven it
is worse, and no shrinkage rule separates them in advance. The honest conclusion: the months are independent draws; use train plus
the scored month's *own earlier days* (causal), and switch a bottleneck only on overwhelming evidence (≥ 10 days, head rate ≥ 0.3/day
→ < 0.05) — a rule that is exactly neutral within train (0.606 vs 0.607).

## 7. Learning the dynamics

Everything below is scored as Task 2 on the same generated *ongoing* windows (organizer rules: ≥ 2 queued cells in the history,
a queue in the horizon, persistence IoU ≤ 0.9), trained Jul–Dec, threshold chosen on January, tested on **February**.

| model | inputs | IoU Feb |
|---|---|---|
| persistence (state at T repeated) | — | 0.74–0.79 |
| hour, v, v₋₁, dv/dt, v ahead, dv/dp, dv/dp further → LightGBM per horizon | 7 | 0.81 (official June) |
| LightGBM, 52 engineered features (lags 5–60 min, ±1…5 links, loads, vehicle counts, road, queue prior) | 52 | 0.85 |
| + density, spacing, Δk per cell | 60 | 0.85 (tie) |
| + typical-week × remainder decomposition inputs | 57 | 0.85 (tie) |
| structural logistic: typical week × position + jam status + remainder | 9 coefficients | 0.71 (= persistence) |
| learned one-step cell rule (own + ±2 neighbours), rolled out 6 steps | ~30 | 0.81 → 0.82 with k and spacing |
| SVD "core" of 9×9 space-time patches + logistic, 5 / 20 / 40 components | — | 0.62 / 0.77 / 0.81 |
| **LightGBM on the raw 9×9 patch** (no engineering) | 162 | **0.86** |
| raw 21×9 patch | 378 | **0.864** |
| ARIMA per detector (best of (0,1,1), (1,1,0), (1,0,0), (2,1,1)) | — | = persistence |
| Kalman tail tracker vs 30-min trend of the tail (MAE km at +30) | — | 0.743 vs 0.754 (persistence 1.04) |

The pattern: the information is in the space-time neighbourhood and it is **non-linear** (a threshold at the front). Linear
machinery (ARIMA, Kalman, SVD cores, logistic) lands at persistence; any tree on the neighbourhood lands at ~0.85–0.86 — with
engineered features or with the raw patch alike. The one-step rule learned from the data uses the neighbours for < 0.5 % of its
splits: on a 5-minute grid a jam rarely crosses a link boundary in one step, so "what comes in comes out" collapses to "what stands,
stands".

## 8. Scoreboard, and the leaderboard decomposition

| hypothesis | status | number |
|---|---|---|
| jams ripple upstream from a fixed bottleneck | **confirmed** | head drift 0; 1,519 rippling episodes; tail 1–4 km/h |
| jams recur by weekday × time (calendar forecast) | rejected | catches 7 % of jams; weekday matters only before 10 h |
| flow in front smaller than flow arriving → jam | confirmed as load, not as Δq | load 0.79 AUC; Δq at the tail 1.004 median, noise |
| speed derivatives announce the jam | rejected | AUC 0.50–0.58 (triangular FD) |
| density / spacing announce it | **confirmed** | 0.76–0.82 AUC, earliest of all; useless inside the selected windows |
| the week-time × position decomposition + remainder | descriptive yes (90 % load, 52 % speed); forecast no | structural logistic = persistence |
| cars on the road set the jam length | **confirmed** | 0.6 km / 100 cars; N₀ = L c/v; R² 0.94–0.98 in March/April |
| capacity collapses when a new bottleneck appears | rejected | discharge/record identical before/after (0.64/0.64 …) |
| March forecasts April | rejected as a rule | RMSE 6.5 vs 5.2 pp; shift correlation −0.07 |
| Kalman / ARIMA / shock formula / CTM / SVD core | rejected | all at or below persistence |
| raw space-time patch + tree | **best ongoing** | 0.864 (21×9) |

**Leaderboard (public = March).** Task scores add with fixed weights, so single-component probes decompose the gap exactly:

| submission | change | public |
|---|---|---|
| own fill (interpolation + front-line jump) + persistence + NNLS ODME | — | 0.7263 |
| + onset = place × time memory, top-4 at T+30 | 320 cells | **0.7930 (+0.067)** |
| + ongoing LightGBM (local +0.10) | 1,492 cells | 0.7947 (+0.002) |
| + onset LightGBM (local +0.03) | 192 cells | 0.7948 (+0.000) |
| state replaced by the public CatBoost ensemble ([anubisyy](https://www.kaggle.com/code/anubisyy/catboost-lgbm-nnls-pipeline-top-50-0-86517)) | all state | 0.8069 (+0.012) |
| queue replaced by the public LightGBM/CatBoost queue | 2,145 cells | 0.8423 (+0.048) |

Two lessons we would not have believed without the board: (1) our generated windows over-estimated *our* ongoing gains ~8×, while the
public team's proxy tracked the board; (2) their queue picks overlap ours in 76–91 % of positive cells and still score 0.16 higher on
S_queue — the 10–25 % of cells at the edges decide IoU. The public ODME (L-BFGS) and our NNLS are identical to the last digit; an
ensemble of their state fill with ours is monotonically worse than theirs alone (0.9416 → 0.9349 at 50/50); adding our eight physics
features to their state model changes nothing (0.9372 vs 0.9371). On Task 1 the public CatBoost is the ceiling we could find;
on Task 2 the edges of the jam are where the competition is still open.

*Scripts (one per figure/table) and the full measurement log `NOTA.md` are in the repository behind this notebook; the figures are in the
attached dataset. Data: the competition release only; organizer code: [trafficflowbench-public](https://github.com/jacky850/trafficflowbench-public).*